In [194]:
# Employee Attrition – SQL Analysis

#This notebook uses SQL queries to analyze employee attrition patterns.
#The goal is to identify high-risk departments, roles, and salary trends
#that contribute to employee turnover.

import pandas as pd
import sqlite3

# Load CSV
df = pd.read_csv("employee_data.csv")

# Create SQLite DB
conn = sqlite3.connect("attrition.db")

# Store table
df.to_sql("employees", conn, index=False, if_exists="replace")


1470

In [75]:
# Employee Attrition – SQL Analysis
#
#Ths notebook uses SQL queries to analyze employee attrition patterns.
#The goal is to identify high-risk departments, roles, and salary trends
#that contribute to employee turnover.


pd.read_sql("PRAGMA table_info(employees);", conn)


,cid,name,type,notnull,dflt_value,pk
0,0,Age,INTEGER,0,None,0
1,1,Attrition,TEXT,0,None,0
2,2,BusinessTravel,TEXT,0,None,0
3,3,DailyRate,INTEGER,0,None,0
4,4,Department,TEXT,0,None,0
5,5,DistanceFromHome,INTEGER,0,None,0
6,6,Education,INTEGER,0,None,0
7,7,EducationField,TEXT,0,None,0
8,8,EmployeeCount,INTEGER,0,None,0
9,9,EmployeeNumber,INTEGER,0,None,0


In [195]:
#This query calculates the attrition percentage for each department, allowing comparison of attrition risk across departments.
query = """
SELECT department,
COUNT(*) AS total_employees,
SUM(CASE WHEN attrition = 'Yes' THEN 1 ELSE 0 END) AS attrition_count
FROM employees
GROUP BY department;
"""

pd.read_sql(query, conn)


,Department,total_employees,attrition_count
0,Human Resources,63,12
1,Research & Development,961,133
2,Sales,446,92


In [196]:
#This query calculates the total number of employees who have left the organization. It gives a high-level view of overall employee turnover.
query ="""
SELECT COUNT(*) AS total_attrition_count
FROM employees
WHERE attrition = 'Yes';
"""
pd.read_sql(query,conn)

,total_attrition_count
0,237


In [72]:
query = """
SELECT
ROUND(
SUM(CASE WHEN attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
2
) AS ATTRITION_RATE
FROM employees;
"""

pd.read_sql(query,conn)

,ATTRITION_RATE
0,16.12


In [48]:
#his analysis shows the total number of employees and the number of employees who left in each department. It helps identify departments with higher attrition counts.
query = """
SELECT department,
COUNT(*) AS TOTAL_EMPLOYEES,
SUM(CASE WHEN attrition = 'Yes' THEN 1 ELSE 0 END) AS TOTAL_ATTRITION_COUNT
FROM employees
GROUP BY department;
"""
pd.read_sql(query,conn)

,Department,TOTAL_EMPLOYEES,TOTAL_ATTRITION_COUNT
0,Human Resources,63,12
1,Research & Development,961,133
2,Sales,446,92


In [43]:
#This query calculates the attrition percentage for each department, allowing comparison of attrition risk across departments.
query = """
SELECT department,
ROUND(
SUM(CASE WHEN attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
2
) AS ATTRITION_RATES
FROM employees
GROUP BY department;
"""
pd.read_sql(query,conn)

,Department,ATTRITION_RATES
0,Human Resources,19.05
1,Research & Development,13.84
2,Sales,20.63


In [198]:
#This query compares the average monthly income of employees who left the company versus those who stayed, helping assess the impact of salary on attrition.query  
query = """
SELECT attrition,
ROUND(AVG(MonthlyIncome),2) AS AVG_SALARY
FROM employees
GROUP BY attrition;
"""
pd.read_sql(query,conn)

,Attrition,AVG_SALARY
0,No,6832.74
1,Yes,4787.09


In [199]:
#his analysis finds the average number of years employees had spent at the company before leaving, providing insight into whether attrition occurs early or later in tenure.
query = """
SELECT ROUND(AVG(YearsAtCompany),2) AS AVG_EXP_OF_STAYED_EMPLOYEES
FROM employees
GROUP BY attrition;
"""
pd.read_sql(query,conn)

,AVG_EXP_OF_STAYED_EMPLOYEES
0,7.37
1,5.13


In [96]:
#This query identifies departments where the attrition rate exceeds a critical threshold (20%), highlighting high-risk departments that need attention.
query = """
SELECT department
FROM employees
GROUP BY department
HAVING
SUM(CASE WHEN attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*) > 20;
"""
pd.read_sql(query,conn)

,Department
0,Sales


In [128]:
#This query lists job roles with the highest number of employees who left, helping identify roles that may have higher workload, stress, or dissatisfaction.
query = """
SELECT JOBROLE,
COUNT(*) AS TOTAL_ATTRITION_COUNT
FROM employees
WHERE attrition = 'Yes'
GROUP BY JOBROLE
ORDER BY TOTAL_ATTRITION_COUNT DESC;
"""
pd.read_sql(query,conn)

,JobRole,TOTAL_ATTRITION_COUNT
0,Laboratory Technician,62
1,Sales Executive,57
2,Research Scientist,47
3,Sales Representative,33
4,Human Resources,12
5,Manufacturing Director,10
6,Healthcare Representative,9
7,Manager,5
8,Research Director,2


In [143]:
#This analysis ranks departments based on their attrition rate using a window function, making it easy to prioritize departments by risk level.
query = """
SELECT department,
ROUND(
SUM(CASE WHEN attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
2
) AS TOTAL_ATTRITION_RATES,
RANK() OVER(
ORDER BY 
SUM(CASE WHEN attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 /COUNT(*),
2
)AS RANK_RATES
FROM employees
GROUP BY department;
"""
pd.read_sql(query,conn)

,Department,TOTAL_ATTRITION_RATES,RANK_RATES
0,Research & Development,13.84,1
1,Human Resources,19.05,2
2,Sales,20.63,3


In [169]:
#This query uses a window function to identify the top three job roles with the highest attrition counts, focusing on the most vulnerable roles.
query = """
SELECT JOBROLE,attrition_count
FROM
(
SELECT JOBROLE,
COUNT(*) AS ATTRITION_COUNT,
ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) 
FROM employees
WHERE attrition = 'Yes'
GROUP BY JOBROLE
)
LIMIT 3;
"""
pd.read_sql(query,conn)

,JOBROLE,ATTRITION_COUNT
0,Laboratory Technician,62
1,Sales Executive,57
2,Research Scientist,47


In [177]:
#This query identifies employees whose salary is below their department’s average salary, helping analyze whether lower pay contributes to attrition risk.
query = """
SELECT EMPLOYEENUMBER,Age,MonthlyIncome,department
FROM employees e
WHERE MonthlyIncome < (
SELECT AVG(MonthlyIncome)
FROM employees
WHERE department = e.department
);

"""
pd.read_sql(query,conn)

,EmployeeNumber,Age,MonthlyIncome,Department
0,1,41,5993,Sales
1,2,49,5130,Research & Development
2,4,37,2090,Research & Development
3,5,33,2909,Research & Development
4,7,27,3468,Research & Development
...,...,...,...,...
982,2060,26,2966,Sales
983,2061,36,2571,Research & Development
984,2064,27,6142,Research & Development
985,2065,49,5390,Sales


In [188]:
query = """
SELECT DISTINCT PerformanceRating
FROM employees;

"""
pd.read_sql(query,conn)

,PerformanceRating
0,3
1,4


In [200]:
#This analysis filters employees with low job satisfaction and low performance ratings, identifying individuals who may be at higher risk of leaving.
query = """
SELECT EmployeeNumber,Age, Department, JobSatisfaction, PerformanceRating
FROM employees
WHERE JobSatisfaction <= 2
  AND PerformanceRating = 3;

"""
pd.read_sql(query,conn)

,EmployeeNumber,Age,Department,JobSatisfaction,PerformanceRating
0,7,27,Research & Development,2,3
1,14,35,Research & Development,2,3
2,20,29,Research & Development,1,3
3,21,32,Research & Development,2,3
4,28,34,Research & Development,2,3
...,...,...,...,...,...
473,2054,29,Research & Development,1,3
474,2055,50,Sales,1,3
475,2057,31,Research & Development,1,3
476,2062,39,Research & Development,1,3
